# 07 — System Integration

**NewsBot Intelligence System 2.0** | ITAI 2373 | Trilok Kalani (SOLO)

Put the modules together into one end-to-end analysis and a mini report.

In [1]:
# --- Setup: works in Colab and locally ---
import os, sys, subprocess

def find_repo_root(start="."):
    p = os.path.abspath(start)
    for _ in range(6):
        if os.path.isdir(os.path.join(p, "src")) and os.path.exists(os.path.join(p, "src", "newsbot.py")):
            return p
        p = os.path.dirname(p)
    return None

ROOT = find_repo_root()
if ROOT is None:
    # Running on a fresh Colab: clone the repo
    if not os.path.isdir("ITAI2373-Portfolio"):
        subprocess.run(["git","clone","--depth","1",
                        "https://github.com/Tikskalani/ITAI2373-Portfolio.git"], check=False)
    ROOT = find_repo_root("ITAI2373-Portfolio/ITAI2373-NewsBot-Final") or \
           find_repo_root("ITAI2373-Portfolio")
sys.path.insert(0, ROOT)
print("Repo root:", ROOT)

# spaCy model (quiet no-op if already present)
try:
    import spacy; spacy.load("en_core_web_sm")
except Exception:
    subprocess.run([sys.executable,"-m","spacy","download","en_core_web_sm"], check=False)

import pandas as pd
DATA = os.path.join(ROOT, "data", "raw", "newsbot_bbc.csv")
df = pd.read_csv(DATA)
print("Loaded", len(df), "articles;", df["category"].nunique(), "categories")
df.head(2)

Repo root: /content/ITAI2373-NewsBot-Final


Loaded 2225 articles; 5 categories


,article_id,category,text
0,business_001,business,Ad sales boost Time Warner profit Quarterly pr...
1,business_002,business,Dollar gains on Greenspan speech The dollar ha...


### One call, full analysis

In [2]:
from src.newsbot import NewsBot
bot = NewsBot(use_domain_rules=True).train(df["text"], df["category"])
import json
out = bot.analyze("A major technology company launched a new AI system, intensifying competition with rivals.")
print(json.dumps(out, indent=2, default=str))

{
  "classification": {
    "category": "tech",
    "confidence": 0.592,
    "runner_up": "business (0.30)",
    "recognized_terms": 8,
    "note": "",
    "key_terms": [
      "launched",
      "rival",
      "competition",
      "major",
      "technology",
      "system"
    ]
  },
  "sentiment": {
    "compound": 0.128,
    "label": "positive",
    "polarity": 0.099,
    "subjectivity": 0.477,
    "emotion": "neutral"
  },
  "entities": [
    [
      "AI",
      "GPE"
    ]
  ]
}


### End-to-end mini pipeline
Classify, summarize, and find related coverage for one article, then format a brief.

In [3]:
from src.language_models.summarizer import Summarizer
from src.language_models.embeddings import SemanticSearch
from src.language_models.generator import ContentGenerator

search = SemanticSearch().index(df["text"])
article = df[df["category"]=="business"]["text"].iloc[0]
analysis = bot.analyze(article)
print("CATEGORY:", analysis["classification"])
print("\nSUMMARY:", Summarizer().summarize(article, n_sentences=2))
print("\nBRIEF:", ContentGenerator().enhance(analysis))
print("\nRELATED:")
for i, score, snippet in search.search(article[:200], k=3):
    print("  -", df["category"].iloc[i], "|", snippet[:60])

CATEGORY: {'category': 'business', 'confidence': 0.974, 'runner_up': 'tech (0.02)', 'recognized_terms': 160, 'note': '', 'key_terms': ['aol', 'warner', 'profit', 'fourth quarter', 'full year', 'revenue']}

SUMMARY: Ad sales boost Time Warner profit Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier. But its film division saw profits slump 27% to $284m, helped by box-office flops Alexander and Catwoman, a sharp contrast to year-earlier, when the third and final film in the Lord of the Rings trilogy boosted results.

BRIEF: This article reads as business news with a positive tone. Key entities include Time Warner, Quarterly, US, TimeWarner, 76%.

RELATED:
  - business | Ad sales boost Time Warner profit Quarterly profits at US me
  - business | US retail sales surge in December US retail sales ended the 
  - business | Sales 'fail to boost High Street' The January sales have fai


**Takeaway.** The facade and modules compose cleanly into a single workflow, which is exactly what the Flask web app calls under the hood.